In [ ]:
!pip install "transformers>=4.40.0" "datasets" "peft" "bitsandbytes" "accelerate" "trl==0.9.4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.7/226.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.2/181.2 kB 7.8 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer, SFTConfig

In [ ]:
MODEL_NAME = "NousResearch/Hermes-3-Llama-3.2-3B"
NEW_MODEL_NAME = "socratic-adapter"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/955 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

In [ ]:
# 4. PREPARE FOR TRAINING
model = prepare_model_for_kbit_training(model)

# Define the LoRA configuration (The "Adapter" settings)
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)


In [ ]:
dataset = load_dataset("json", data_files="/content/socratic_train.json", split="train")

def format_row(example):
    # Combine user instruction and assistant output into one Llama 3 formatted string
    text = (f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
            f"You are a Socratic Tutor. Answer with questions.<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n\n"
            f"{example['instruction']}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            f"{example['output']}<|eot_id|>")
    return {"text": text}

# Apply the formatting immediately
dataset = dataset.map(format_row)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/47 [00:00<?, ? examples/s]

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text", # Now this works because we created the column
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=TrainingArguments(
        output_dir="./results",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        weight_decay=0.001,
        fp16=True,
        logging_steps=1,
        optim="paged_adamw_32bit",
    ),
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:269: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:307: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override

Map:   0%|          | 0/47 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:402: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


In [ ]:
print("Starting Training...")
trainer.train()
print("Training Finished.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128039, 'pad_token_id': 128039}.


Starting Training...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,6.264600
2,5.204400
3,4.999900
4,4.590000
5,4.123000
6,4.069300


Training Finished.


In [ ]:
# 7. SAVE ADAPTER
trainer.model.save_pretrained(NEW_MODEL_NAME)
tokenizer.save_pretrained(NEW_MODEL_NAME)
print(f"Adapter saved to: {NEW_MODEL_NAME}")

Adapter saved to: socratic-adapter


In [ ]:
import os
from google.colab import files
print("⏳ Step 5: Setting up conversion tools...")
if not os.path.exists("llama.cpp"):
    !git clone https://github.com/ggerganov/llama.cpp
    !pip install -r llama.cpp/requirements.txt

print("⚙️ Converting to GGUF...")
# Run the conversion script
!python llama.cpp/convert_lora_to_gguf.py socratic-adapter --outfile socratic-adapter.gguf

# -----------------------------------------------------------------
# 8. DOWNLOAD
# -----------------------------------------------------------------
print("🎉 DONE! Checking for file...")
if os.path.exists("socratic-adapter.gguf"):
    file_size = os.path.getsize("socratic-adapter.gguf") / (1024 * 1024)
    print(f"✅ socratic-adapter.gguf created successfully ({file_size:.2f} MB)")
    print("⬇️ Downloading now...")
    files.download("socratic-adapter.gguf")
else:
    print("❌ Error: GGUF file was not created. Check logs above.")


⏳ Step 5: Setting up conversion tools...
Cloning into 'llama.cpp'...
remote: Enumerating objects: 76762, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 76762 (delta 1), reused 2 (delta 1), pack-reused 76759 (from 1)
Receiving objects: 100% (76762/76762), 282.02 MiB | 30.98 MiB/s, done.
Resolving deltas: 100% (55539/55539), done.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.3 MB/s eta 0:00

⚙️ Converting to GGUF...
INFO:lora-to-gguf:Loading base model from Hugging Face: NousResearch/Hermes-3-Llama-3.2-3B
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:lora-to-gguf:Exporting model...
INFO:hf-to-gguf:blk.0.ffn_down.weight.lora_a, torch.float32 --> F32, shape = {8192, 16}
INFO:hf-to-gguf:blk.0.ffn_down.weight.lora_b, torch.float32 --> F32, shape = {16, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight.lora_a, torch.float32 --> F32, shape = {3072, 16}
INFO:hf-to-gguf:blk.0.ffn_gate.weight.lora_b, torch.float32 --> F32, shape = {16, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight.lora_a,  torch.float32 --> F32, shape = {3072, 16}
INFO:hf-to-gguf:blk.0.ffn_up.weight.lora_b,  torch.float32 --> F32, shape = {16, 8192}
INFO:hf-to-gguf

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
from google.colab import files
print("🎉 DONE! Checking for file...")
if os.path.exists("socratic-adapter.gguf"):
    file_size = os.path.getsize("socratic-adapter.gguf") / (1024 * 1024)
    print(f"✅ socratic-adapter.gguf created successfully ({file_size:.2f} MB)")
    print("⬇️ Downloading now...")
    files.download("socratic-adapter.gguf")
else:
    print("❌ Error: GGUF file was not created. Check logs above.")

🎉 DONE! Checking for file...
✅ socratic-adapter.gguf created successfully (92.78 MB)
⬇️ Downloading now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
import os
# Download the GGUF file (This is the one you really need for the app)
# Note: Large files might take a while or fail if internet is unstable.
if os.path.exists("socratic-adapter.gguf"):
    files.download("socratic-adapter.gguf")
else:
    print("Please run the conversion script (Phase 2.5) first!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>